# Neutron Reflectometry — Normalizing Flow Training Demo

This notebook demonstrates the training process for a normalizing-flow (NF) model used for neutron reflectometry inference. We use the config **`nf_config_mixed_sigmas_qweighted_quick.yaml`**, which trains an NF model with:

- **dR (measurement uncertainties) as input** to the network
- **Mixed data** — a blend of synthetic on-the-fly reflectivity curves and experimental data
- **Q-weighted input transformations** — curves are scaled as $R' = R \cdot Q^{-\alpha}$ ($\alpha = 2$) and sigmas as $dR' = \frac{dR \cdot Q^{-\beta}}{R \cdot \ln(10)}$ ($\beta = 3$)

> **Note:** This is for illustration purposes only. The training runs for very few iterations with a small batch size (~15 min max) and **does not produce a reasonable model**.

## 1. Imports

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import yaml
from pathlib import Path
from collections import defaultdict

from reflectorch.runs.config import load_config
from reflectorch.runs.utils import (
    get_trainer_from_config,
    get_paths_from_config,
    get_callbacks_from_config,
    train_from_config,
)

torch.manual_seed(42)
np.random.seed(42)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

## 2. Load and Inspect the Training Configuration

The configuration file `nf_config_mixed_sigmas_qweighted_quick.yaml` defines the full training setup. Let's load it and review its key sections.

In [ ]:
config_name = 'nf_config_mixed_sigmas_qweighted_quick.yaml'
config = load_config(config_name)

print(yaml.dump(config, default_flow_style=False, sort_keys=False))

### Key configuration highlights

| Section | Setting | Value | Meaning |
|---------|---------|-------|---------|
| **Dataset** | `cls` | `MixedReflectivityDataLoader` | Blends synthetic + experimental data |
| | `mix_fraction` | 0.5 | 50% experimental, 50% synthetic |
| | `curves_scaler.cls` | `QWeightedCurvesScaler` | $R' = R \cdot Q^{-2}$ |
| | `sigma_scaler.cls` | `QWeightedSigmaScaler` | $dR' = dR \cdot Q^{-3} / (R \cdot \ln 10)$ |
| **Model** | `embedding_net.in_channels` | 3 | R + Q + σ (three-channel input) |
| | `flow_kwargs.num_layers` | 40 | Number of coupling layers |
| **Training** | `num_iterations` | 100 | Very few iterations (quick demo) |
| | `batch_size` | 32 | Small batch size |
| | `lr` | 1e-4 | Learning rate |
| | `trainer_cls` | `NFlowTrainer` | NF-specific trainer (negative log-prob loss) |

## 3. Initialize the Trainer

The `get_trainer_from_config` function:
1. Creates the **data loader** (`MixedReflectivityDataLoader`) with Q-weighted scalers
2. Builds the **NF network** (`NFNetwork` with a convolutional embedding net + rational-quadratic spline flow)
3. Configures the **optimizer** (AdamW) and wraps everything in an `NFlowTrainer`

In [ ]:
trainer = get_trainer_from_config(config)

num_params = sum(p.numel() for p in trainer.model.parameters())
print(f'Model parameters: {num_params / 1e6:.2f} M')
print(f'Trainer class:    {type(trainer).__name__}')
print(f'Loader class:     {type(trainer.loader).__name__}')
print(f'Batch size:       {trainer.batch_size}')

## 4. Inspect a Training Batch

Before training, let's look at a single batch to understand the data pipeline. The `MixedReflectivityDataLoader` produces batches with:
- `scaled_noisy_curves` — Q-weighted reflectivity curves
- `q_values` — momentum transfer values
- `scaled_sigmas` — Q-weighted measurement uncertainties
- `scaled_params` — scaled parameters + subprior bounds
- `q_resolutions` — resolution parameters

In [ ]:
batch = trainer.loader.get_batch(trainer.batch_size)

print('Batch keys:', list(batch.keys()))
for key, val in batch.items():
    if isinstance(val, torch.Tensor):
        print(f'  {key:25s} shape={str(list(val.shape)):15s} dtype={val.dtype}  device={val.device}')
    else:
        print(f'  {key:25s} type={type(val).__name__}')

## 5. Setup Callbacks

We initialize the training callbacks from the config. For this quick demo, model saving is disabled. The LR scheduler uses cosine annealing with a short warmup.

In [ ]:
folder_paths = get_paths_from_config(config, mkdir=True)
callbacks = get_callbacks_from_config(config, folder_paths)

print(f'Model save path:  {folder_paths["model"]}')
print(f'Losses save path: {folder_paths["losses"]}')
print(f'Callbacks: {callbacks}')

## 6. Run Training

The NF training loop:
1. Samples a batch of (synthetic + experimental) reflectivity data
2. Applies Q-weighted transformations to curves and sigmas
3. Feeds the 3-channel input (R, Q, σ) + prior bounds through the embedding net
4. Computes the negative log-probability under the conditional normalizing flow
5. Backpropagates and updates parameters

The loss function is:
$$\mathcal{L} = -\frac{1}{B} \sum_{i=1}^{B} \log p_\theta(\boldsymbol{\theta}_i \mid \mathbf{x}_i)$$

where $\boldsymbol{\theta}_i$ are the physical parameters and $\mathbf{x}_i$ is the conditioned input.

In [ ]:
num_iterations = config['training']['num_iterations']
update_tqdm_freq = config['training']['update_tqdm_freq']
grad_accumulation_steps = config['training'].get('grad_accumulation_steps', 1)

print(f'Starting training for {num_iterations} iterations...')
print(f'  Batch size:              {trainer.batch_size}')
print(f'  Learning rate:           {config["training"]["lr"]}')
print(f'  Grad accumulation steps: {grad_accumulation_steps}')
print(f'  Clip grad norm max:      {config["training"]["clip_grad_norm_max"]}')
print()

In [ ]:
trainer.train(
    num_iterations,
    callbacks=callbacks,
    disable_tqdm=False,
    use_notebook_tqdm=True,
    update_tqdm_freq=update_tqdm_freq,
    grad_accumulation_steps=grad_accumulation_steps,
)

print(f'\nTraining finished! Final loss: {trainer.losses["total_loss"][-1]:.4f}')

## 7. Visualize Training Results

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

losses = trainer.losses['total_loss']

# Raw loss
ax.plot(losses, linewidth=1, alpha=0.4, color='blue', label='Raw loss')

# Smoothed loss (running average)
window = min(10, len(losses))
if window > 1:
    smoothed = np.convolve(losses, np.ones(window)/window, mode='valid')
    ax.plot(range(window-1, len(losses)), smoothed, linewidth=2, color='red', label=f'Smoothed (window={window})')

ax.set_xlabel('Iteration', fontsize=14)
ax.set_ylabel('Loss (negative log-prob)', fontsize=14)
ax.set_title('Training Loss', fontsize=16)
ax.legend(fontsize=12)
ax.grid(alpha=0.2)

plt.tight_layout()
plt.show()

print(f'Initial loss: {losses[0]:.4f}')
print(f'Final loss:   {losses[-1]:.4f}')
print(f'Best loss:    {min(losses):.4f} (at iteration {np.argmin(losses)})')

## 8. Save Model Weights

We save the trained model weights so they can be loaded for inference (even though this demo model is undertrained).

In [ ]:
model_path = folder_paths['model']

torch.save({'model': trainer.model.state_dict()}, model_path)
print(f'Model saved to: {model_path}')

# Also save training metadata
losses_path = folder_paths['losses']
torch.save({
    'paths': folder_paths,
    'losses': dict(trainer.losses),
    'lrs': trainer.lrs,
    'params': config,
}, losses_path)
print(f'Losses saved to: {losses_path}')

## 9. Quick Inference Test (Sanity Check)

Even though the model is heavily undertrained, we can verify that the full predict → sample pipeline works. We generate a synthetic batch and run inference on it.

In [ ]:
# Generate a test batch
trainer.model.eval()

with torch.no_grad():
    test_batch_data = trainer.loader.get_batch(1)
    test_batch = trainer.get_batch_by_size(1)
    
    # Sample from the flow
    num_samples = 50
    samples = trainer.model.sample(
        num_samples=num_samples,
        curves=test_batch.scaled_curves.expand(num_samples, -1),
        bounds=test_batch.scaled_bounds.expand(num_samples, -1) if test_batch.scaled_bounds is not None else None,
        q_values=test_batch.scaled_q_values.expand(num_samples, -1) if test_batch.scaled_q_values is not None else None,
        sigmas=test_batch.scaled_sigmas.expand(num_samples, -1) if test_batch.scaled_sigmas is not None else None,
        conditioning_params=test_batch.scaled_conditioning_params.expand(num_samples, -1) if test_batch.scaled_conditioning_params is not None else None,
    )

samples_np = samples.detach().cpu().numpy()

print(f'Sampled parameters shape: {samples_np.shape}')
print(f'All finite: {np.isfinite(samples_np).all()}')
print(f'\nSample statistics:')
print(f'{"Param dim":<12} | {"Mean":>10} | {"Std":>10} | {"Min":>10} | {"Max":>10}')
print('-' * 60)
for i in range(samples_np.shape[1]):
    print(f'{"dim " + str(i):<12} | {samples_np[:, i].mean():>10.4f} | {samples_np[:, i].std():>10.4f} | {samples_np[:, i].min():>10.4f} | {samples_np[:, i].max():>10.4f}')

## 10. Summary

This notebook demonstrated the **end-to-end training workflow** for a normalizing flow model in reflectorch:

1. **Configuration** — Loaded a YAML config defining the full training setup
2. **Data pipeline** — `MixedReflectivityDataLoader` blends experimental and synthetic data, with Q-weighted input transformations ($\alpha=2, \beta=3$)
3. **Model** — `NFNetwork` with a convolutional embedding net and 40-layer rational-quadratic spline flow
4. **Training** — `NFlowTrainer` minimizes the negative log-probability $-\log p_\theta(\boldsymbol{\theta} | \mathbf{x})$
5. **Monitoring** — Loss curves and learning rate schedules
6. **Inference** — Verified the sampling pipeline works on the trained model

> **For a properly trained model**, increase `num_iterations` (e.g. 50,000–100,000), use a larger `batch_size` (e.g. 512), and provide a representative experimental dataset in `dataset/train/`.